# 第7回: RAG

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session07/session07_rag.ipynb)

LLMは人間のような文章を生成できるが、その知識は学習データに固定されており、リアルタイム情報・社内データ・高度に専門的な情報にはアクセスできない。**RAG (Retrieval-Augmented Generation)** はこの制約に対処するパターンで、外部の最新かつ文脈固有の情報をLLMに統合し、出力の正確性・関連性・事実性を高める。AIエージェントにとっては、静的な学習知識を超えて検証可能なデータに行動と応答を根拠付けられることが重要で、RAGはエージェントを単なる会話相手からデータ駆動で実務をこなすツールへと変える。

---
## 0. 環境準備

In [ ]:
%pip install -q langchain langchain-core langchain-openai langchain-text-splitters langgraph numpy matplotlib matplotlib-fontja

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY を入力する: ")

print("APIキー設定完了" if os.environ.get("OPENAI_API_KEY") else "未設定")

---
# 1. RAG パターンの概要

RAGは、応答を生成する前に**外部の知識ベースへアクセスする**ことでLLMの能力を大きく拡張するパターン。事前学習された内部知識だけに頼るのではなく、人間が本を調べたりインターネットを検索したりするように、LLMが情報を「調べる」ことを可能にする。

ユーザーの質問はLLMに直接送られるのではなく、次の流れで処理される。

1. **Retrieval（検索）**: まず外部知識ベース（文書・DB・Webページなどを整理した「ライブラリ」）から関連情報を探す。これは単純なキーワード一致ではなく、ユーザーの意図と言葉の意味を理解する**セマンティック検索 (semantic search)**、つまり「質問と意味的に関連する部分」を膨大な知識ベースから自動で選び出す検索技術。最も関連の深い情報の断片（**チャンク**）が取り出される
2. **Augmentation（拡張）**: 取り出したチャンクを元のプロンプトに追加し、より情報量の多いクエリを作る
3. **Generation（生成）**: 拡張されたプロンプトをLLMに送る。LLMには、重みを更新しなくてもプロンプト内に与えられた情報をその場で根拠として使える**文脈内学習 (In-Context Learning)** という性質があり、これにより流暢なだけでなく検索されたデータに事実として根拠付けられた応答を生成できる

つまり、次のように整理するとよい。

$$\text{RAG} = \text{Semantic Search} + \text{In-Context Learning}$$

特にRAGを支えるコア技術はRetrieval (Semantic Search)にある。

> 実務では埋め込みに基づく Semantic Search が主流だが、Retrieval の方法はそれだけではない。キーワード検索（BM25 など）や、両者を組み合わせたハイブリッド検索も使われる（詳細は後述）。

RAGの主な利点:

- 知識ベースを更新するだけで、再学習なしに学習データ以降の情報を回答へ反映できる
- 応答を検証可能なデータに根拠付けることで**ハルシネーション**（誤情報の生成）のリスクを減らせる
- 社内文書やWikiにある**専門知識**を活用できる
- 情報の出所を示す**引用（citation）** が可能になり、信頼性と検証可能性が高まる

[![](https://mermaid.ink/img/pako:eNqtVMtO20AU_ZXRrFrVIY6NCXGlqkDYwSJV1UViFoM9xhbOOBo7ojREIp4NlD5WDWLVB22FQNAFUheIlo-ZGoW_6NjBYBe1q84i8dx7ztx7zh27B03fwlCHtuevmQ6iIVh4YlCDALHqvhk87oHAQR2sA0vsJOChZezpwIDx5-EVO-Bsm0fHnH3h7CNnp5xtGbCf8YPu8gpFHQf8OtuJt19fng3jaC9LJstyKTZD1ye5ollhUCo9AnMtzgac7SdHR9_iV8OlPGxujJkvgPjgJH6_w6PB6Oc5H1wUCfMpY8MlAabhBnhWn23d42wvobEtzo7qs_dvCJhYyeMdMZdvDkZHw8u9KH-y6XTJas6sDi26dTX8dLW5n-8z51PRiqez-XijVebsK2fnPPqe_A5ORqeH8bu35aVUSkOIF81HYhCH_1LeuFYe4DYioWuCACNqOqkFeZXJEqEUPNaUzzRvBZouNT2cU_jAgKCgKP98Yx5nu5wdpzdlN7F8cDS6-BG__JAHp5XygetJFL1OW2z-rWDjbrqZhhYWFls20m1Uov6yH4JFcf29glUCkSJnWuUZEqxhWv7zSqRNeCgI6tgGiLhtFGIQhNRfxSULCYMoRes6qEmalIv6th3gUIRlWRqTxLTFayVSQNEC4LlEzAS4xHaJG-KHhUoAV7JKWfx__0MJrlDXgnpIu1iCbUzbKNnCXpI3YOjgNjZgMmsL0VUDGqQvOB1Emr7fzmjU76442abbsUTDdReJ4d8ihIuYzvldEkJdrSlqegbUe_A51EuTtYkpVa2qlUpNU6bVqiLBdQGT5YmqIiuaKmvatFqp9iX4Iq1amZCnFHVSnqxVa1VZm65IEFtu6NPF8Xct_bz1fwMqOrbJ?type=png)](https://mermaid.live/edit#pako:eNqtVMtO20AU_ZXRrFrVIXaMSeJKVYGwg0WqqovELAZ7jC2ccTR2RGmI1Hg2UPpYNYhVH7QVAkEXSF0gWj5mahT-omMHU7uoXXUWiefec-bec-7YfWj6FoY6tD1_3XQQDcHiI4MaBIjV8M3gYR8EDupiHVhiJwEPrWBPBwaMP42u2AFn2zw65uwzZx84O-Vsy4CDjB_0VlYp6jrg59lOvP3q8mwUR3tZMlmWS7EZuj7JFc0Kg1LpAZhvczbkbD85Ovoavxwt52HzE8xCAcSHJ_G7HR4Nxz_O-fCiSFhIGZsuCTANN8GTxlz7Dmd7CY1tcXbUmLt7Q8DESh5vibl8fTA-Gl3uRfmTTadH1nJmdWnRravRx6vn-_k-cz4VrXg8l48322XOvnB2zqNvye_wZHx6GL99U15OpTSFeNF8JAZx-C_lzWvlAe4gEromCDCippNakFeZLBFKwRNN-Uzrt0DTpaaHcwrvGRAUFOWfb8zjbJez4_Sm7CaWD4_GF9_jF-_z4LRSPnA9iaLXaYutvxVs3k630tDi4lLbRrqNStRf8UOwJK6_V7BKIFLkbLs8S4J1TMt_Xom0CQ8FQQPbABG3g0IMgpD6a7hkIWEQpWhDB3VJk3JR37YDHIqwLEsTkpi2eK1EClS0AHguETMBLrFd4ob4fqESwEpWKYv_738owVXqWlAPaQ9LsINpByVb2E_yBgwd3MEGTGZtIbpmQIMMBKeLSMv3OxmN-r1VJ9v0upZouOEiMXyBsJEXJBBhI6bzfo-EUFfrSi09BOp9-BTqpen61IyqVlVFqWuVmlqtSHBDwGR5qlqRK5oqa1pNVaoDCT5LyypT8kxFnZan69V6VdZqigSx5YY-XZp82NLv2-AX5gy3GA)

---
# 2. 文脈内学習（In-Context Learning）

LLMには **文脈内学習（In-Context Learning, ICL）** と呼ばれる性質がある。**重み（パラメータ）を一切更新せずに**、プロンプト（コンテキストウィンドウ）内に与えられた情報や例示をその場で利用して振る舞いを変えられる、というもの。「学習」といってもファインチューニングのようなパラメータ更新ではなく、**プロンプト内の情報を一時的な作業記憶として使う**というイメージ。

代表的な形態:

- **Zero-shot**: 指示だけを与える（例示から学習しているわけではないため、定義によってはICLに含めないこともある）
- **Few-shot**: 少数の入出力例を与えると、その場でパターンを読み取って模倣する
- **知識の注入**: 学習データに無い事実や文書をプロンプトに与えると、それを根拠に回答できる

このうち Few-shot と知識の注入をコードで確認する。

In [ ]:
# @title Few-shot: プロンプト内の例示だけで振る舞いが変わる
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
    seed=42,
)

# 指示は一切書かず、入出力の例だけを与える
few_shot_prompt = """\
入力: りんご 3個 120円
出力: {"item": "りんご", "qty": 3, "unit_price": 120}

入力: 牛乳 2本 258円
出力: {"item": "牛乳", "qty": 2, "unit_price": 258}

入力: 食パン 1斤 189円
出力:"""

# 重みは一切更新していないのに、プロンプト内の例からパターンを読み取って模倣する
response = llm.invoke(few_shot_prompt)
print(response.content)

In [ ]:
# @title 知識の注入: Retrievalなしの「手動Augmentation」

question = "架空通貨『ゾルタ』は1ゾルタ何円ですか？"

# 文脈なし: 学習データに無い情報なので答えられない
print("--- 文脈なし ---")
print(llm.invoke(question).content)

# 文脈を人手でプロンプトに貼り付ける（Augmentation → Generation だけを再現）
context = "社内レポート抜粋: 架空通貨ゾルタの本日の為替レートは 1ゾルタ = 152円 である。"

augmented_prompt = f"""次の文脈だけを根拠に質問に答えてください。

文脈: {context}

質問: {question}"""

print("\n--- 文脈あり（手動Augmentation） ---")
print(llm.invoke(augmented_prompt).content)

このように、知識を注入すれば学習データに無いことにも答えられる。ただし、次の点には注意が必要。

- **プロンプトが大きくなりすぎると性能が悪化する可能性がある**。コンテキストウィンドウには上限があるうえ、上限内であっても入力が長くなるほど文脈の中間にある情報を見落としやすくなる（いわゆる "lost in the middle"）。トークン数の増加はコストとレイテンシにも直結する
- **「知識の注入」にノイズを含めるべきではない**。質問と無関係な情報が文脈に混ざると、モデルがそれに引きずられて回答品質が下がる。注入する文脈は、答えたい質問に関連するものだけに絞る

したがって、手元に大量の文書（社内規程集・製品マニュアルなど）があっても丸ごとプロンプトへ注入するわけにはいかず、**質問に関連する部分だけを選び出して注入する仕組み**、すなわち **Retrieval** が必要になる。

---
# 3. RAGのコア概念

| 概念 | 役割 |
|---|---|
| **埋め込み（Embeddings）** | テキストを意味を捉えた数値ベクトルに変換する |
| **意味的類似度** | 2つのテキストがどれだけ「同じことを言っているか」を測る |
| **チャンク化（Chunking）** | 大きな文書を検索しやすい小さな断片に分割する |
| **ベクトルデータベース** | 埋め込みを保存し、意味に基づく高速検索を提供する |

以降、それぞれをコードで確認する。

## 3.1 埋め込み（Embeddings）

埋め込みとは、単語・フレーズ・文書といったテキストの**数値表現**で、ベクトル（数値のリスト）の形をとる。テキストの意味と、テキスト同士の関係を数学的な空間に射影する。**意味が近いテキストは、このベクトル空間内で近い位置に置かれる。**

単純な2次元のグラフを想像すると、「猫」が座標 (2, 3) なら「子猫」はすぐ近くの (2.1, 3.1) に、意味の異なる「自動車」は (8, 1) のような離れた座標になるイメージ。実際の埋め込みは数百〜数千次元の空間にあり、言語のニュアンスを非常に細かく捉えられる。

[![](https://mermaid.ink/img/pako:eNpVkLFOwzAQhl8lujmq3DZpWq_phngBCIOFTRLR2OVwJErVIenIwIDEiioxwMKMUNU-jIVa3gLHVYrw5Ps-_2f75nCpuAAKNyXjyKSOM4Y6wUR6dulcT4T3_fJg6mq_WZtqu1-97R43u_evn-cnU32Y-tUsV2a5NvVnG4qZpt456fR8j3Siixaf5FrPDqLfmMHgqMYqdaLvRPgnYoZODBvedRh8SDHnQDWWwodCYMGaEuZNJAGdiUIkQO2WM7xOIJELm5kyeaZU0cZQlWkG9IpNbm1VTjnTYpyzFFlxpCgkFxirUmqgI0JcE6BzuAMadu0ze0E0CIdBEAWEhD7MLB51yH-88OHe3Ws_EdlDguda4elh5m70i1_FKYGx?type=png)](https://mermaid.live/edit#pako:eNpVkL1OwzAQx18lujmK0jRfeE03xAuAGU7YJFGbuByORIkyhJWBAYkVscHCAyBUHqZCLW-B4ypFePL9fvf3x7VwoYQEBlcNCsJaZwWS5sRrxyxd6oV0vp_vN3f9bv256b92L6_bh_X27ePn6XHTv4-NGWrmnPle4Dq-l5yP-LjUerUX08HE8UHNVG7F1IroT2RIVqQDn1gMLuRUCmCaGulCJanCoYR2iHDQhawkB2a2AmnOgdedySyxPlWqGmOkmrwAdomLa1M1S4FazkrMCasDJVkLSZlqag0sTQJ7CLAWboBFE_PMIEziKA3DJPT9yIWVwUee_x93Ltzae80nEtMkRakVneznbMfd_QL0aHlq)

以下のコードは、`OpenAIEmbeddings` で「King」「Queen」「Man」「Woman」を含む複数の単語の埋め込み（`text-embedding-3-small` では1536次元）を**PCA（主成分分析）によって第1・第2主成分の2次元空間に射影**して図示する。向きがテキストの意味を表す。

あわせて、埋め込み空間に意味の演算が成り立つことを示す有名な例として、**`Queen = King − Man + Woman`** が成り立つことを確認している。「Man → Woman」という差分（性別の方向）を King に足すと、結果は Queen の近くに現れる。これは意味の関係が空間内の一定方向として表現されていることを示している。

> このような意味的関係式は、word2vec / GloVe のような古典的な単語埋め込みで有名になった性質で、現代の OpenAI 埋め込みは、単語単体よりも文章・段落・チャンクの意味表現に向いているため、この種のアナロジーが完全にきれいに出るとは限らない。


In [ ]:
# @title 埋め込みを作って図示してみる
import matplotlib.pyplot as plt
import matplotlib_fontja  # matplotlibで日本語を表示できるようにする
import numpy as np
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small" # @param ["text-embedding-3-small", "text-embedding-3-large"]
)

analogy_words = ["King", "Queen", "Man", "Woman"]
extra_words = [
    "Prince",
    "Princess",
    "Boy",
    "Girl",
    "Mother",
    "Father",
    "Dog",
    "Car",
]
words = analogy_words + extra_words
vectors = embeddings.embed_documents(words)
king_vector, queen_vector, man_vector, woman_vector = vectors[:4]

# 各テキストが高次元のベクトル（数値のリスト）になる
for word, vec in zip(words, vectors):
    print(f"{word}: 次元数={len(vec)} 先頭5要素={[round(v, 4) for v in vec[:5]]}")

# アナロジー演算: King - Man + Woman を計算し、単位長に正規化する
analogy_vector = np.array(king_vector) - np.array(man_vector) + np.array(woman_vector)
analogy_vector /= np.linalg.norm(analogy_vector)

def cosine_similarity(a: list[float] | np.ndarray, b: list[float] | np.ndarray) -> float:
    """2つのベクトルのコサイン類似度を返す。"""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Queen と King - Man + Woman のコサイン類似度: {cosine_similarity(queen_vector, analogy_vector):.3f}")
print(f"Man と Woman のコサイン類似度: {cosine_similarity(man_vector, woman_vector):.3f}")
print(f"King と Queen のコサイン類似度: {cosine_similarity(king_vector, queen_vector):.3f}")

# 高次元のままでは図示できないので、PCA（主成分分析）で2次元に圧縮して散布図にする
matrix = np.array(vectors)
mean = matrix.mean(axis=0)
centered = matrix - mean
_, _, v_t = np.linalg.svd(centered, full_matrices=False)
points_2d = centered @ v_t[:2].T  # 第1・第2主成分に射影
analogy_points_2d = points_2d[:4]
extra_points_2d = points_2d[4:]

# アナロジーベクトルも同じ主成分空間に射影する
analogy_2d = (analogy_vector - mean) @ v_t[:2].T

fig, ax = plt.subplots(figsize=(6, 5))

# 原点から各点への矢印として、2次元に射影されたベクトルの向きを図示する
ax.quiver(
    np.zeros(len(analogy_points_2d)),
    np.zeros(len(analogy_points_2d)),
    analogy_points_2d[:, 0],
    analogy_points_2d[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1,
    color="tab:blue",
    alpha=0.35,
    width=0.005,
)
ax.quiver(
    0,
    0,
    analogy_2d[0],
    analogy_2d[1],
    angles="xy",
    scale_units="xy",
    scale=1,
    color="tab:red",
    alpha=0.6,
    width=0.006,
)
for word, (x, y) in zip(analogy_words, analogy_points_2d):
    ax.annotate(word, (x, y), fontsize=14, xytext=(8, 8), textcoords="offset points")

# 追加した単語・文章はPCAの計算には含めるが、矢印ではなくマーカーだけで表示する
ax.scatter(extra_points_2d[:, 0], extra_points_2d[:, 1], s=45, color="gray", alpha=0.7)
for word, (x, y) in zip(extra_words, extra_points_2d):
    ax.annotate(word, (x, y), fontsize=10, color="gray", xytext=(5, 5), textcoords="offset points")

ax.annotate(
    "King - Man + Woman",
    analogy_2d,
    fontsize=12,
    color="tab:red",
    xytext=(8, -14),
    textcoords="offset points",
)

all_points = np.vstack([points_2d, analogy_2d, [0, 0]])
x_min, y_min = all_points.min(axis=0)
x_max, y_max = all_points.max(axis=0)
x_margin = max((x_max - x_min) * 0.25, 0.05)
y_margin = max((y_max - y_min) * 0.25, 0.05)
ax.set_xlim(x_min - x_margin, x_max + x_margin)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_title("埋め込みの2次元図示（意味が近い単語ほど近くに置かれる）")
ax.set_xlabel("第1主成分")
ax.set_ylabel("第2主成分")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.2 意味的類似度（Semantic Similarity）

Retrieval では、ユーザーの質問と知識ベース内の各断片を比べ、「どれが関連深いか」を数値で決める必要がある。この指標となるのが**意味的類似度**、つまり使われている単語そのものではなく**意味と文脈**に基づいて2つのテキストの近さを測る考え方。

意味的類似度が高いと、言い回しが違っても同じ内容なら近いと判断できる。例えば次のペアはいずれも意味的には近い。

- 「フランスの首都は？」と「フランスの首都はどの都市ですか？」（言い回しが違うだけの同じ質問）

これによって、ユーザーの言い回しが知識ベースの記述と一致しなくても関連情報を見つけ出せる。なお、単語の重なりだけで測る古典的な**字句的類似度**（BM25 などのキーワード検索）もあるが、上のような言い換えは苦手で、現代の RAG では意味的類似度が主流となっている。

埋め込みベクトル同士の意味的類似度は、**コサイン類似度**（ベクトルのなす角のコサイン。1に近いほど意味が近い）で計算するのが一般的。値が大きいほど意味が近く、逆に**意味的距離**は類似度が高いほど小さくなる。

In [ ]:
# @title コサイン類似度で意味の近さを測る
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """2つの埋め込みベクトルのコサイン類似度（-1〜1、1に近いほど意味が近い）を返す。"""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


pairs = [
    # 言い回しが違うだけの同じ質問 → 高い類似度
    ("フランスの首都は？", "フランスの首都はどの都市ですか？"),
    # 意味が違う → 低い類似度
    ("フランスの首都は？", "おすすめのラーメン屋を教えて"),
    ("猫", "自動車"),
]

for text_a, text_b in pairs:
    vec_a, vec_b = embeddings.embed_documents([text_a, text_b])
    score = cosine_similarity(vec_a, vec_b)
    print(f"{score:.3f}  「{text_a}」×「{text_b}」")

## 3.3 チャンク化（Chunking）

チャンク化は、大きな文書を扱いやすい小さな断片（**チャンク**）に分割する処理。RAGシステムは巨大な文書を丸ごとLLMに渡すのではなく、この小さなチャンク単位で処理する。分割の仕方は情報の文脈と意味を保つうえで重要。

例えば50ページのユーザーマニュアルを1つのテキスト塊として扱うのではなく、セクション・段落・文といった単位に分割する。「トラブルシューティング」のセクションは「インストールガイド」とは別のチャンクになる。ユーザーが特定の問題について質問したとき、RAGシステムはマニュアル全体ではなく最も関連するトラブルシューティングのチャンクだけを取得できるため、検索は速くなり、LLMに渡す情報もユーザーの必要により合致したものになる。

### 主なチャンク化戦略

どう分割するかで検索品質は大きく変わる。代表的な戦略は次の通り。

| 戦略 | 分割の基準 | 特徴 |
|---|---|---|
| **固定サイズ分割（Fixed-size）** | 文字数・トークン数で機械的に区切る | 実装が最も簡単で高速。ただし文や段落の途中で切れて文脈が壊れやすい |
| **再帰的分割（Recursive）** | 段落→文→単語…と、区切り文字を優先順位付きで試す | 固定サイズの手軽さを保ちつつ、意味の切れ目を尊重しやすい。実務の既定値として広く使われる（本ノートの `RecursiveCharacterTextSplitter`） |
| **構造的分割（Structure-aware）** | 見出し・章・Markdownやコードの構文など文書構造 | 「第2章 経費精算規程」のような論理単位で切れる。構造が明確な文書（マニュアル・規程・HTML/Markdown・ソースコード）に有効 |
| **意味的分割（Semantic）** | 隣接文の埋め込み類似度が下がる箇所＝話題の変わり目 | 話題ごとにまとまった質の高いチャンクになりやすい。埋め込み計算のコストと処理時間が増える |

一般には「再帰的分割を既定にし、文書の構造がはっきりしていれば構造的分割を併用、検索品質を突き詰めたい場合に意味的分割を検討する」という選び方になる。チャンクは大きすぎるとノイズが増え、小さすぎると文脈が失われるため、サイズの調整も重要。

### オーバーラップ（重複）

分割の境界で文脈が途切れるのを防ぐため、**隣接するチャンクの端を一定量だけ重複させる**テクニックがオーバーラップ。

重複を増やすほど文脈は途切れにくくなるが、似た内容の重複チャンクが取得されやすくもなる。目安として `chunk_size` の 10〜20% 程度に設定することが多い（本ノートでは `chunk_size=200` に対し `chunk_overlap=40`）。


In [ ]:
# @title 知識ベースとなる文書を用意してチャンク化する
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LLMが学習していない架空の社内規程（エンタープライズ検索・Q&Aのユースケースを想定）
company_policy = """\
従業員ハンドブック 社内規程集（サンプル）

第1章 リモートワーク規程
リモートワーク（在宅勤務）は全正社員を対象とし、週3日まで利用できる。
利用する場合は前営業日の17時までに勤怠システムで申請する。
コアタイムは10時から15時とし、この時間帯はオンラインで連絡が取れる状態を維持する。
入社後3か月間の試用期間中はリモートワークを利用できない。

第2章 経費精算規程
1件5,000円以上の物品購入は、購入前に上長の承認を得る必要がある。
交通費は最も経済的な経路で精算する。新幹線の利用は片道100km以上の出張に限る。
領収書は支出日の属する月の月末から5営業日以内に経費精算システムへ提出する。
提出期限を過ぎた場合、原則として精算できない。

第3章 有給休暇規程
年次有給休暇は入社6か月経過後に10日付与し、以後勤続年数に応じて最大20日まで付与する。
有給休暇は半日単位でも取得できる。取得予定日の3営業日前までに申請することを推奨する。
付与から2年で失効するため、計画的な取得を推奨する。

第4章 情報セキュリティ規程
業務データは会社が管理するクラウドストレージにのみ保存し、私物のUSBメモリの使用を禁止する。
社外で業務用PCを使用する場合は、必ずVPNを経由して社内ネットワークに接続する。
インシデント（紛失・漏えい等）が発生した場合は、1時間以内に情報システム部へ報告する。
"""

# RecursiveCharacterTextSplitter は「再帰的分割（Recursive）」を行うチャンキングクラス。
# 区切り文字リスト（既定は ["\n\n", "\n", " ", ""] ＝段落→行→単語→文字）を優先順位順に試し、
# 段落など大きな意味の切れ目でまず分割する。それでも chunk_size を超える断片は、
# 次に優先度の高い区切り文字で再帰的に分割し直す。こうして「できるだけ意味のまとまりを保ちつつ、
# 各チャンクを chunk_size 以下に収める」ため、実務の既定チャンカーとして広く使われる。
# chunk_size: 各チャンクの最大文字数 / chunk_overlap: 文脈が途切れないよう隣接チャンクと重複させる文字数
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, # @param {type:"integer"}
    chunk_overlap=40, # @param {type:"integer"}
)

chunks = text_splitter.split_documents(
    [Document(page_content=company_policy, metadata={"source": "社内規程集"})]
)

print(f"チャンク数: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"--- chunk {i} ({len(chunk.page_content)}文字) ---")
    print(chunk.page_content[:60], "...")

## 3.4 ベクトルデータベース

ベクトルデータベースは、埋め込みを効率よく保存・検索するために設計された特殊なデータベース。チャンク化された文書は埋め込みに変換され、この高次元ベクトルとして保存される。

ベクトルデータベースは**セマンティック検索専用に作られており**、ユーザーのクエリもベクトルに変換したうえで、HNSW（Hierarchical Navigable Small World）のような最適化アルゴリズムで数百万のベクトルから意味的に最も「近い」ものを高速に探し出す。**他の手法が「単語」を探すのに対し、ベクトルデータベースは「意味」を探す。**

実装形態はさまざま:

- マネージドDB: Pinecone, Weaviate
- OSS: Chroma DB, Milvus, Qdrant
- 既存DBへのベクトル検索拡張: Redis, Elasticsearch, Postgres（pgvector）
- 中核の検索ライブラリ: Meta AI の FAISS、Google Research の ScaNN

ここでは追加のインフラ不要でノートブック内で完結する `InMemoryVectorStore`（LangChain組み込みのインメモリ実装）を使う。

In [ ]:
# @title チャンクを埋め込んでベクトルストアに保存し、セマンティック検索する
from langchain_core.vectorstores import InMemoryVectorStore

# チャンクを埋め込みに変換してベクトルストアへ保存（Indexing）
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

# 知識ベースの文言（「リモートワーク」「在宅勤務」）と違う言い回しでも意味で検索できる
query = "家で仕事できるのは週に何回まで？" # @param {type:"string"}

results = vectorstore.similarity_search_with_score(query, k=2)

print(f"クエリ: {query}\n")
for doc, score in results:
    print(f"[類似度スコア: {score:.3f}]")
    print(doc.page_content)
    print("-" * 80)

---
# 4. RAGパイプラインの実装（LangChain + LangGraph）

コア概念が揃ったので、RAGパイプライン全体を LangGraph の `StateGraph` で実装する。構成は前回までと同じく **State / Node / Edge** で、次の2つのノードを直列につなぐ。

- `retrieve`: ユーザーの質問をクエリにベクトルストアを検索し、関連チャンクを取得する（**Retrieval**）
- `generate`: 取得したチャンクをプロンプトに差し込み（**Augmentation**）、LLMに回答を生成させる（**Generation**）

[![](https://mermaid.ink/img/pako:eNpVj7EKwjAQhl8lZGpAoU1N2jgIgm7ioG7GITYXW7AVQqKD-O6e0Q7e8N0Px33HPWlzs0DnhLrr7dG0xgey2emBYO0Py90hy1JjjEynC-Ih-A7ucByD1oOOShiOzHmpoxR1hZSu1rHMJWBWRaNjZTk_fb3jbjJeYABvAhzHkIz_FoV2WZcKySFP3rOOQlq0V2cxQ4rCfaZF_rsx2tKN9XaVZQjG6ITQHnxvOos_P2looU_fW3AmXgN9vd4RgFT2?type=png)](https://mermaid.live/edit#pako:eNpVj7EKwjAQhl8lZGpAoU1N2jgIgm7ioG7GITYXW7AVQqKD-O6e0Q7e8N0Px33HPWlzs0DnhLrr7dG0xgey2emBYO0Py90hy1JjjEynC-Ih-A7ucByD1oOOShiOzHmpoxR1hZSu1rHMJWBWRaNjZTk_fb3jbjJeYABvAhzHkIz_FoV2WZcKySFP3rOOQlq0V2cxQ4rCfaZF_rsx2tKN9XaVZQjG6ITQHnxvOos_P2looU_fW3AmXgN9vd4RgFT2)

まずは比較のため、RAG無しのLLMに社内規程について質問してみる。

In [ ]:
# @title RAG無しで質問してみる（LLMは社内規程を知らない）
response = llm.invoke("この会社ではリモートワークは週何日まで利用できますか？")
print(response.content)

In [ ]:
# @title State（状態）
from typing import Annotated, TypedDict

# TypedDict は、決まったキーと値の型を持つ辞書を表す型
class RAGGraphState(TypedDict):
    # ユーザーの質問
    question: str

    # 検索で取得した関連チャンク
    documents: list[Document]

    # LLMが生成した回答
    generation: str

In [ ]:
# @title retrieve node（Retrieval: 検索）

# ベクトルストアをRetriever（検索インターフェース）として使う
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3, # 取得するチャンクの最大件数（SIMILARITY_TOP_K に相当）
    }
)

def retrieve_documents(state: RAGGraphState) -> RAGGraphState:
    """ユーザーの質問に基づいて関連チャンクを検索する。"""
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question, "generation": ""}

In [ ]:
# @title generate node（Augmentation + Generation: 拡張と生成）
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 検索した文脈（Context）を質問と一緒にプロンプトへ差し込む
RAG_PROMPT_TEMPLATE = """あなたは質問応答タスクのアシスタントです。
以下の検索された文脈（Context）を使って質問に答えてください。
答えが分からない場合は、分からないと答えてください。
最大3文で、簡潔に答えてください。

質問: {question}

Context: {context}

回答:
"""


def generate_response(state: RAGGraphState) -> RAGGraphState:
    """検索されたチャンクを根拠にLLMで回答を生成する。"""
    question = state["question"]
    documents = state["documents"]

    prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

    # 検索されたチャンクを連結してContextを作る（Augmentation）
    context = "\n\n".join(doc.page_content for doc in documents)

    # RAGチェーン: プロンプト -> LLM -> 文字列に変換
    rag_chain = prompt | llm | StrOutputParser()

    generation = rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "documents": documents, "generation": generation}

In [ ]:
# @title Edges & Graph
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(RAGGraphState)

# nodes
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_response)

# edges
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

rag_app = workflow.compile()

In [ ]:
# @title RAGパイプラインを実行する

# @markdown 質問（RAG無しでは答えられなかった質問）
query = "リモートワークは週何日まで利用できますか？" # @param {type:"string"}

for step in rag_app.stream({"question": query}, stream_mode="updates"):
    for node_name, update in step.items():
        print(f"=== node: {node_name} ===")
        if "documents" in update and node_name == "retrieve":
            for doc in update["documents"]:
                print("  [検索されたチャンク]", doc.page_content[:50], "...")
        if update.get("generation"):
            print("  [生成された回答]", update["generation"])
    print("-" * 80)

In [ ]:
# @title 別の質問でも試す

# @markdown 質問
query_2 = "新幹線を使っていいのはどんなとき？" # @param {type:"string"}

result = rag_app.invoke({"question": query_2})
print("質問:", query_2)
print("回答:", result["generation"])
print()

# @markdown 知識ベースに無い質問（分からないと答えることを確認）
query_3 = "社員食堂のメニューを教えて" # @param {type:"string"}

result = rag_app.invoke({"question": query_3})
print("質問:", query_3)
print("回答:", result["generation"])

---
# 5. Agentic RAG

ここまでのRAG（Naive RAG）は「**必ず検索して、取れたものをそのまま使う**」受動的なパイプライン。このパターンの進化形が **Agentic RAG** で、検索の前後に推論と意思決定のレイヤーを追加し、情報抽出の信頼性を大きく高める。エージェントは検索結果を受動的に受け入れるのではなく、その品質・関連性・完全性を能動的に吟味する**ゲートキーパー**として振る舞う。

Agentic RAGでエージェントが担う役割の例:

1. **内省とソースの検証**: 「リモートワークの規定は？」に対し、2020年の古いブログ記事と2025年の正式な規程が両方ヒットした場合、メタデータから最新の正式文書を選び、古い方を捨ててからLLMに渡す
2. **知識の矛盾の解消**: 「プロジェクトAlphaのQ1予算は？」に対し、初期提案書（5万ユーロ）と確定財務報告書（6.5万ユーロ）が矛盾していたら、信頼性の高い財務報告書を優先する
3. **マルチステップ推論**: 「自社と競合Xの機能と価格を比較して」を、自社機能・自社価格・競合機能・競合価格の4つのサブクエリに分解し、個別に検索してから統合する
4. **知識ギャップの検出と外部ツールの利用**: 内部知識ベースに情報が無ければ、Web検索などの外部ツールを起動して補う

ここでは最小のAgentic RAGとして「**検索するかどうかをLLM自身が判断する**」構成を実装する。検索を**ツール**としてエージェントに渡し、前回までのReActループ（`tools_condition`）で制御する。雑談には検索を使わず、知識が必要な質問のときだけ検索ツールを呼ぶ。

[![](https://mermaid.ink/img/pako:eNqlUE2LwjAU_CshpxYUtrGfHgRBb9WDejOLPJNXK7YNtAkerP_dJArL7nVzmAzDy8zkPahQEumc0KpRd1FDr0m54x2xZ39Y7g5B4K8wJNPpgsAFO330yHlXlhtuZl-x4CYtIouZZMwpKXKTF3nETRKz1CmF1RPGYjuZgPx-B3gf5zu6ERE7rIRDPLunmefZzPKoAseTzAf6kBxGopVqhuOA0Iv6dOvUvUF5wdMZBrT9_tb6nfAp4S1-Pve_ZumHj2S9XQWBhTCkE0Jb7Fu4SrvnB9U1tn7jEiswjabP5wsQynik?type=png)](https://mermaid.live/edit#pako:eNqlUE2LwjAU_CshpxYUtrGfHgRBb9WDejOLPJNXK7YNtAkerP_dJArL7nVzmAzDy8zkPahQEumc0KpRd1FDr0m54x2xZ39Y7g5B4K8wJNPpgsAFO330yHlXlhtuZl-x4CYtIouZZMwpKXKTF3nETRKz1CmF1RPGYjuZgPx-B3gf5zu6ERE7rIRDPLunmefZzPKoAseTzAf6kBxGopVqhuOA0Iv6dOvUvUF5wdMZBrT9_tb6nfAp4S1-Pve_ZumHj2S9XQWBhTCkE0Jb7Fu4SrvnB9U1tn7jEiswjabP5wsQynik)

In [ ]:
# @title 検索をツール化してエージェントに渡す
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool(parse_docstring=True)
def search_knowledge_base(query: str) -> str:
    """社内規程（リモートワーク・経費精算・有給休暇・情報セキュリティ）を検索して関連する条文を返す。

    Args:
        query: 検索したい内容。例 'リモートワークの上限日数'。
    """
    documents = retriever.invoke(query)
    if not documents:
        return "関連する規程は見つかりませんでした。"
    return "\n\n".join(doc.page_content for doc in documents)


tools = [search_knowledge_base]

model_with_tools = llm.bind_tools(tools)


class AgentState(TypedDict):
    # 会話履歴（add_messages で新しいメッセージが積み上がる）
    messages: Annotated[list, add_messages]


system_message = SystemMessage(
    content=(
        "あなたは社内規程を案内するアシスタントです。"
        "社内規程・制度に関する質問には、推測せず必ず search_knowledge_base ツールで根拠を検索してから、"
        "検索結果に基づいて答えてください。検索結果に無いことは分からないと答えてください。"
        "検索が不要な雑談や一般的な質問には、ツールを使わずそのまま答えてください。"
    )
)


def agent_node(state: AgentState) -> AgentState:
    """検索の要否をLLM自身が判断するノード。必要ならツール呼び出しを要求する。"""
    response = model_with_tools.invoke([system_message] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")

# LLMがツール呼び出しを要求したら tools へ、しなければ END へ
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    },
)

builder.add_edge("tools", "agent")

agentic_rag_app = builder.compile()

In [ ]:
# @title Agentic RAGを実行する: 知識が必要な質問（検索ツールが呼ばれる）

# @markdown 質問
question = "5,000円のキーボードを買いたいのですが、何か手続きは必要ですか？" # @param {type:"string"}

for chunk in agentic_rag_app.stream(
    {"messages": [HumanMessage(content=question)]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        for message in update["messages"]:
            message.pretty_print()

In [ ]:
# @title Agentic RAGを実行する: 検索が不要な質問（ツールは呼ばれない）

# @markdown 質問
question = "こんにちは！あなたは何ができますか？" # @param {type:"string"}

for chunk in agentic_rag_app.stream(
    {"messages": [HumanMessage(content=question)]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        for message in update["messages"]:
            message.pretty_print()

---
# 6. RAGの課題と発展形

## 6.1 RAGの課題

強力なパターンだが、課題も残る。

- **情報の分散**: 答えに必要な情報が1つのチャンクに収まらず、文書内の複数箇所や複数文書に散らばっている場合、検索器が必要な文脈をすべて集めきれず、不完全・不正確な回答になり得る
- **検索品質への依存**: チャンク化と検索の品質にシステム全体の性能が左右される。無関係なチャンクが取れるとノイズになりLLMを混乱させる
- **矛盾する情報の統合**: 相反するソースからの情報を適切に統合するのは依然として難しい
- **前処理と運用のコスト**: 知識ベース全体をベクトルDB等へ事前処理・格納する必要があり、社内Wikiのような更新され続けるソースでは定期的な同期も必要
- **性能への影響**: レイテンシ・運用コスト・プロンプトのトークン数が増える

## 6.2 GraphRAG

**GraphRAG** は、ベクトルDBの代わりに**ナレッジグラフ**を使う発展形。データのエンティティ（ノード）間の明示的な関係（エッジ）をたどって複雑な質問に答える。複数文書に断片化した情報の統合という従来RAGの弱点に強く、金融分析（企業と市場イベントの関連付け）や科学研究（遺伝子と疾患の関係発見）などに使われる。一方で、高品質なナレッジグラフの構築・維持には大きなコストと専門性が必要で、柔軟性が低くレイテンシも増えやすい。

## 6.3 まとめ

- RAGは、**Retrieval（検索）→ Augmentation（拡張）→ Generation（生成）** の流れで、LLMに外部の最新・固有情報へのアクセスを与えるパターン
- 技術要素としては **RAG = Semantic Search + In-Context Learning** と整理できる。Augmentation → Generation は**文脈内学習（In-Context Learning）** の応用でRAGとは独立に成立する。RAGの技術的核心は、注入する文脈を知識ベースから自動で見つけ出す**Retrieval（Semantic Search）**にある
- 埋め込み・意味的類似度・チャンク化・ベクトルDBがRAGを支えるコア技術。キーワード検索（BM25）やハイブリッド検索と組み合わせることもある
- RAGは学習データの陳腐化やハルシネーションを抑え、根拠の**引用**により検証可能な回答を実現する
- **Agentic RAG** は検索の周りに推論レイヤーを追加し、ソースの検証・矛盾の解消・クエリ分解・外部ツール活用によって回答の信頼性と深さを高める。その代償として複雑さ・レイテンシ・コストが増える
- 適用場面: 学習データに無い固有・最新・専門の情報に基づいて回答させたい場合。社内文書Q&A、カスタマーサポート、引用付きの事実ベース応答など

RAGはLLMを「クローズドブック」の会話相手から、外部知識を参照できる「オープンブック」の推論ツールへと変える、エージェント構築における重要パターン。